# Mixed-traffic Fast-DeePC

Fixed benchmark: $T=1200$. This notebook runs the original hard-equality DeeP-LCC, the full soft quadratic problem with OSQP, and the genuinely $T$-independent C Fast-DeePC controller. It reports raw mean online solve times and writes one GIF comparing the original and Fast-DeePC trajectories.


## 1. Import packages and locate the repository


In [ ]:
from pathlib import Path
import ctypes as ct
import os
import shutil
import subprocess
import time

# Avoid severe BLAS oversubscription when CVXPY, SciPy, and the C DLL coexist.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import scipy.linalg as la
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
HERE = ROOT

print("Repository root:", ROOT)
print("CVXPY solvers:", cp.installed_solvers())


## 2. Build and load the C dense BoxQP solver


In [ ]:
CORE_SOURCE = ROOT / "src" / "deepc_boxqp_optimized.c"
WRAPPER_SOURCE = ROOT / "src" / "deepc_boxqp_optimized_ctypes_wrapper.c"
DLL = ROOT / "build" / "deep_lcc_dense_boxqp.dll"
DLL.parent.mkdir(exist_ok=True)
OPENBLAS_BIN = Path(r"C:\msys64\ucrt64\bin")
OPENBLAS_LIB = Path(r"C:\msys64\ucrt64\lib")

gcc_candidates = [
    shutil.which("gcc"),
    r"C:\msys64\ucrt64\bin\gcc.exe",
]
gcc = next((Path(p) for p in gcc_candidates if p and Path(p).exists()), None)
if gcc is None:
    raise FileNotFoundError("gcc was not found. Install MSYS2/UCRT64 gcc.")

source_mtime = max(CORE_SOURCE.stat().st_mtime, WRAPPER_SOURCE.stat().st_mtime)
needs_build = (not DLL.exists()) or DLL.stat().st_mtime < source_mtime
if needs_build:
    if (OPENBLAS_LIB / "libopenblas.dll.a").exists():
        command = [
            str(gcc), "-O3", "-shared", "-Wl,--export-all-symbols",
            "-o", str(DLL), str(CORE_SOURCE),
            f"-L{OPENBLAS_LIB}", "-lopenblas", "-lm",
        ]
    else:
        print("OpenBLAS was not found; using the slower standalone wrapper.")
        command = [
            str(gcc), "-O3", "-shared", "-Wl,--export-all-symbols",
            "-o", str(DLL), str(WRAPPER_SOURCE), "-lm",
        ]
    print("Compiling:", " ".join(command))
    subprocess.run(command, cwd=HERE, check=True)

if OPENBLAS_BIN.exists() and hasattr(os, "add_dll_directory"):
    _openblas_dll_directory = os.add_dll_directory(str(OPENBLAS_BIN))
lib = ct.CDLL(str(DLL))
BINT = ct.c_long

class BoxQPInfo(ct.Structure):
    _fields_ = [
        ("iterations", BINT),
        ("mu", ct.c_double),
        ("duality", ct.c_double),
        ("status", ct.c_int),
        ("chol_info", BINT),
    ]

DOUBLE_PTR = ct.POINTER(ct.c_double)
lib.dense_boxqp_solve.argtypes = [
    DOUBLE_PTR, DOUBLE_PTR, BINT, ct.c_double, BINT,
    DOUBLE_PTR, DOUBLE_PTR, ct.POINTER(BoxQPInfo),
]
#Hessian matrix H linear term h Problem dimension n. Convergence tolerance.Maximum number of IPM iterations.Optional warm-start solution. Python passes None when no warm start exists.Output array receiving the optimizer z, 
lib.dense_boxqp_solve.restype = ct.c_int

def ptr(array):
    return np.asarray(array, dtype=np.float64).ctypes.data_as(DOUBLE_PTR)

print("Loaded:", DLL)


## 3. Define the fixed platoon and Fast-DeePC parameters


In [ ]:
DT = 0.05
V_STAR = 15.0
S_STAR = 20.0
V_MAX = 30.0
S_ST = 5.0

ID = np.array([0, 0, 1, 0, 0, 1, 0, 0], dtype=int)
N_VEHICLES = len(ID)
POS_CAV = np.flatnonzero(ID == 1)
N_CAV = len(POS_CAV)
DIM_U, DIM_E, DIM_Y = N_CAV, 1, N_VEHICLES + N_CAV

T_INI = 20
HORIZON = 50
T_DATA = 1200                 # paper default; repository brake script uses 2000
CONTROL_TIME = 16.0          # covers brake, hold, recovery, and settling
CONTROL_STEPS = int(CONTROL_TIME / DT)
CONTROL_SHIFT = 10            # solve every 0.5 s; set to 1 for exact paper protocol

WEIGHT_V = 1.0
WEIGHT_S = 0.5
WEIGHT_U = 0.1
LAMBDA_G = 10.0              # paper default; repository brake script uses 100
LAMBDA_Y = 1.0e4

# Fast soft formulation. These enforce the equalities approximately.
LAMBDA_CONSISTENCY = 1.0e4
RHO_FUTURE_OUTPUT = 1.0e4

U_MIN, U_MAX = -5.0, 2.0
SPACING_ERROR_MIN = 5.0 - S_STAR
SPACING_ERROR_MAX = 40.0 - S_STAR
VELOCITY_ERROR_MIN, VELOCITY_ERROR_MAX = -V_STAR, V_MAX - V_STAR

OFFLINE_NOISE = 0.1
ONLINE_NOISE = 0.1
SEED = 42

# Time-major output ordering: [v1,...,v8,s_CAV3,s_CAV6] at each step.
Q_STEP = np.r_[np.full(N_VEHICLES, WEIGHT_V), np.full(N_CAV, WEIGHT_S)]
Q_BAR = np.tile(Q_STEP, HORIZON)
R_BAR = np.full(HORIZON * DIM_U, WEIGHT_U)
SPACING_INDEX = np.concatenate([
    np.arange(k * DIM_Y + N_VEHICLES, (k + 1) * DIM_Y)
    for k in range(HORIZON)
])

print("CAV positions (1-based):", POS_CAV + 1)
print(f"g dimension at T={T_DATA}:", T_DATA - T_INI - HORIZON + 1)
print("Reduced BoxQP dimension:", HORIZON * (DIM_U + DIM_Y))


## 4. Define the nonlinear traffic plant and collect the fixed offline data


In [ ]:
class OVMVehicle:
    def __init__(self, alpha, beta, s_go, dt=DT):
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.s_go = float(s_go)
        self.dt = float(dt)

    def optimal_velocity(self, spacing):
        if spacing <= S_ST:
            return 0.0
        if spacing >= self.s_go:
            return V_MAX
        phase = np.pi * (spacing - S_ST) / (self.s_go - S_ST)
        return 0.5 * V_MAX * (1.0 - np.cos(phase))

    def acceleration(self, velocity, spacing, lead_velocity, noise=0.0):
        desired = self.optimal_velocity(spacing)
        value = self.alpha * (desired - velocity)
        value += self.beta * (lead_velocity - velocity)
        value += float(noise)
        if spacing <= S_ST:
            value = U_MIN
        return float(np.clip(value, U_MIN, U_MAX))


# Table-I heterogeneous parameters, with nominal entries at CAV positions.
ALPHA = np.array([0.45, 0.75, 0.60, 0.70, 0.50, 0.60, 0.40, 0.80])
BETA  = np.array([0.60, 0.95, 0.90, 0.95, 0.75, 0.90, 0.80, 1.00])
S_GO  = np.array([38.0, 31.0, 35.0, 33.0, 37.0, 35.0, 39.0, 34.0])


class MixedTrafficPlatoon:
    def __init__(self, vehicle_id=ID):
        self.ID = np.asarray(vehicle_id, dtype=int)
        self.n = len(self.ID)
        self.pos_cav = np.flatnonzero(self.ID == 1)
        self.hdvs = [OVMVehicle(ALPHA[i], BETA[i], S_GO[i]) for i in range(self.n)]
        self.reset()

    def reset(self):
        self.v = np.full(self.n, V_STAR, dtype=np.float64)
        # Each heterogeneous HDV starts at its own OVM equilibrium; CAV spacing is 20 m.
        self.s = S_ST + (S_GO - S_ST) / np.pi * np.arccos(1.0 - 2.0 * V_STAR / V_MAX)
        self.s = self.s.astype(np.float64)
        self.s[self.pos_cav] = S_STAR

    def output(self):
        velocity_error = self.v - V_STAR
        spacing_error = self.s[self.pos_cav] - S_STAR
        return np.r_[velocity_error, spacing_error]

    def step(self, u_cav, v_head, acceleration_noise=None):
        u_cav = np.asarray(u_cav, dtype=np.float64).reshape(-1)
        if acceleration_noise is None:
            acceleration_noise = np.zeros(self.n)

        old_v = self.v.copy()
        old_s = self.s.copy()
        new_v = old_v.copy()
        new_s = old_s.copy()

        cav_lookup = {vehicle: j for j, vehicle in enumerate(self.pos_cav)}
        for i in range(self.n):
            lead_velocity = float(v_head if i == 0 else old_v[i - 1])
            if self.ID[i] == 1:
                acceleration = float(np.clip(u_cav[cav_lookup[i]], U_MIN, U_MAX))
            else:
                acceleration = self.hdvs[i].acceleration(
                    old_v[i], old_s[i], lead_velocity, acceleration_noise[i]
                )
            new_v[i] = np.clip(old_v[i] + DT * acceleration, 0.0, V_MAX)
            new_s[i] = old_s[i] + DT * (lead_velocity - old_v[i])

        self.v, self.s = new_v, new_s
        return self.output()


def brake_profile(number_of_steps):
    # Repository Experiment C: -5 m/s^2 for 2 s, hold 5 s, +2 m/s^2 for 5 s.
    velocity = np.empty(number_of_steps + HORIZON, dtype=np.float64)
    acceleration = np.zeros_like(velocity)
    velocity[0] = V_STAR
    for k in range(len(velocity) - 1):
        t = k * DT
        acceleration[k] = U_MIN if t < 2.0 else (0.0 if t < 7.0 else (U_MAX if t < 12.0 else 0.0))
        velocity[k + 1] = np.clip(velocity[k] + DT * acceleration[k], 0.0, V_STAR)
    return velocity, acceleration


def collect_offline_data(T, seed=SEED):
    rng = np.random.default_rng(seed)
    platoon = MixedTrafficPlatoon(ID)
    u_data = rng.uniform(-2.0, 2.0, size=(T, DIM_U))
    e_data = rng.uniform(-1.0, 1.0, size=(T, 1))
    noise = rng.uniform(-OFFLINE_NOISE, OFFLINE_NOISE, size=(T, N_VEHICLES))
    y_data = np.empty((T, DIM_Y), dtype=np.float64)
    for k in range(T):
        y_data[k] = platoon.step(u_data[k], V_STAR + e_data[k, 0], noise[k])
    return u_data, e_data, y_data


def block_hankel(data, depth):
    data = np.asarray(data, dtype=np.float64)
    if data.ndim == 1:
        data = data[:, None]
    T, dimension = data.shape
    columns = T - depth + 1
    if columns <= 0:
        raise ValueError(f"T={T} is shorter than Hankel depth {depth}")
    H = np.empty((depth * dimension, columns), dtype=np.float64)
    for j in range(columns):
        H[:, j] = data[j:j + depth].reshape(-1)
    return H


def build_hankel_blocks(u_data, e_data, y_data):
    depth = T_INI + HORIZON
    Hu = block_hankel(u_data, depth)
    He = block_hankel(e_data, depth)
    Hy = block_hankel(y_data, depth)
    return {
        "Up": Hu[:T_INI * DIM_U],
        "Uf": Hu[T_INI * DIM_U:],
        "Ep": He[:T_INI * DIM_E],
        "Ef": He[T_INI * DIM_E:],
        "Yp": Hy[:T_INI * DIM_Y],
        "Yf": Hy[T_INI * DIM_Y:],
    }


u_data, e_data, y_data = collect_offline_data(T_DATA)
BLOCKS = build_hankel_blocks(u_data, e_data, y_data)
print({name: matrix.shape for name, matrix in BLOCKS.items()})


## 5. Build the original, soft OSQP, and $T$-independent C controllers


In [ ]:
def _assign(parameter, value):
    parameter.value = np.asarray(value, dtype=np.float64).reshape(-1)


class OriginalDeepLCC:
    def __init__(self, blocks):
        self.B = blocks
        nc = blocks["Up"].shape[1]
        ny = HORIZON * DIM_Y
        self.g = cp.Variable(nc)
        self.uini = cp.Parameter(T_INI * DIM_U)
        self.eini = cp.Parameter(T_INI * DIM_E)
        self.yini = cp.Parameter(T_INI * DIM_Y)
        self.ef = cp.Parameter(HORIZON * DIM_E)
        self.r = cp.Parameter(ny)

        # Official-style g-only formulation. These expressions eliminate the separate u, y, and sigma_y variables without changing the problem.
        self.u_expr = blocks["Uf"] @ self.g
        self.y_expr = blocks["Yf"] @ self.g
        self.sigma_expr = blocks["Yp"] @ self.g - self.yini

        objective = 0.5 * cp.sum(
            cp.multiply(Q_BAR, cp.square(self.y_expr - self.r))
        )
        objective += 0.5 * cp.sum(
            cp.multiply(R_BAR, cp.square(self.u_expr))
        )
        objective += LAMBDA_G * cp.sum_squares(self.g)
        objective += LAMBDA_Y * cp.sum_squares(self.sigma_expr)

        constraints = [
            blocks["Up"] @ self.g == self.uini,
            blocks["Ep"] @ self.g == self.eini,
            blocks["Ef"] @ self.g == self.ef,
            self.u_expr >= U_MIN,
            self.u_expr <= U_MAX,
            self.y_expr[SPACING_INDEX] >= SPACING_ERROR_MIN,
            self.y_expr[SPACING_INDEX] <= SPACING_ERROR_MAX,
        ]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)
        if not self.problem.is_dpp():
            raise RuntimeError("Original g-only DeeP-LCC problem is not DPP")

    def solve(self, uini, eini, yini, ef, reference):
        _assign(self.uini, uini)
        _assign(self.eini, eini)
        _assign(self.yini, yini)
        _assign(self.ef, ef)
        _assign(self.r, reference)

        start = time.perf_counter()
        solver_used = "CLARABEL"
        try:
            self.problem.solve(
                solver=cp.CLARABEL,
                warm_start=True,
                tol_gap_abs=1e-5,
                tol_gap_rel=1e-5,
                tol_feas=1e-5,
                max_iter=500,
                verbose=False,
            )
        except cp.SolverError:
            solver_used = "OSQP"
            self.problem.solve(
                solver=cp.OSQP,
                warm_start=True,
                eps_abs=1e-4,
                eps_rel=1e-4,
                max_iter=10000,
                polishing=False,
                adaptive_rho=True,
                check_termination=25,
                verbose=False,
            )

        elapsed = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(
                f"Original DeeP-LCC status: {self.problem.status} "
                f"with {solver_used}"
            )

        return {
            "u": np.asarray(self.u_expr.value).reshape(HORIZON, DIM_U),
            "y": np.asarray(self.y_expr.value).reshape(HORIZON, DIM_Y),
            "objective": float(self.problem.value),
            "time": elapsed,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
            "solver": solver_used,
        }

class SoftDeepLCC:
    def __init__(self, blocks):
        self.B = blocks
        nc = blocks["Up"].shape[1]
        nu, ny = HORIZON * DIM_U, HORIZON * DIM_Y
        self.g, self.u, self.y = cp.Variable(nc), cp.Variable(nu), cp.Variable(ny)
        self.uini = cp.Parameter(T_INI * DIM_U)
        self.eini = cp.Parameter(T_INI * DIM_E)
        self.yini = cp.Parameter(T_INI * DIM_Y)
        self.ef = cp.Parameter(HORIZON * DIM_E)
        self.r = cp.Parameter(ny)

        objective = 0.5 * cp.sum(cp.multiply(Q_BAR, cp.square(self.y - self.r)))
        objective += 0.5 * cp.sum(cp.multiply(R_BAR, cp.square(self.u)))
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Up"] @ self.g - self.uini)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Ep"] @ self.g - self.eini)
        objective += LAMBDA_Y * cp.sum_squares(blocks["Yp"] @ self.g - self.yini)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Uf"] @ self.g - self.u)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Ef"] @ self.g - self.ef)
        objective += 0.5 * RHO_FUTURE_OUTPUT * cp.sum_squares(blocks["Yf"] @ self.g - self.y)
        objective += LAMBDA_G * cp.sum_squares(self.g)

        y_low = np.tile(np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MIN), np.full(N_CAV, SPACING_ERROR_MIN)], HORIZON)
        y_high = np.tile(np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MAX), np.full(N_CAV, SPACING_ERROR_MAX)], HORIZON)
        constraints = [self.u >= U_MIN, self.u <= U_MAX, self.y >= y_low, self.y <= y_high]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)

    def solve(self, uini, eini, yini, ef, reference):
        _assign(self.uini, uini); _assign(self.eini, eini); _assign(self.yini, yini)
        _assign(self.ef, ef); _assign(self.r, reference)
        start = time.perf_counter()
        self.problem.solve(
        solver=cp.OSQP,
        warm_start=True,
        eps_abs=1e-4,
        eps_rel=1e-4,
        max_iter=20000,
        polishing=False,
        verbose=False,
        )
        elapsed = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Soft DeeP-LCC status: {self.problem.status}")
        return {
            "u": np.asarray(self.u.value).reshape(HORIZON, DIM_U),
            "y": np.asarray(self.y.value).reshape(HORIZON, DIM_Y),
            "g": np.asarray(self.g.value),
            "objective": float(self.problem.value),
            "time": elapsed,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
        }


class ReducedCDeepLCC:
    """Reduced BoxQP with every T-dependent operation performed offline."""

    def __init__(self, blocks):
        Up, Ep, Yp, Uf, Ef, Yf = (
            blocks[name] for name in ("Up", "Ep", "Yp", "Uf", "Ef", "Yf")
        )
        nc = Up.shape[1]
        nu = HORIZON * DIM_U
        ny = HORIZON * DIM_Y

        # Eliminate g offline.
        M = LAMBDA_CONSISTENCY * (
            Up.T @ Up + Ep.T @ Ep + Uf.T @ Uf + Ef.T @ Ef
        )
        M += 2.0 * LAMBDA_Y * (Yp.T @ Yp)
        M += RHO_FUTURE_OUTPUT * (Yf.T @ Yf)
        M += 2.0 * LAMBDA_G * np.eye(nc)
        M_factor = la.cho_factor(M, lower=True, check_finite=False)

        # Map fixed-size online data directly to the BoxQP linear term.
        # w = col(u_ini, e_ini, y_ini, e_f), whose size does not depend on T.
        C = np.hstack([
            LAMBDA_CONSISTENCY * Up.T,
            LAMBDA_CONSISTENCY * Ep.T,
            2.0 * LAMBDA_Y * Yp.T,
            LAMBDA_CONSISTENCY * Ef.T,
        ])
        P = la.cho_solve(M_factor, C, check_finite=False)
        self.Su = -LAMBDA_CONSISTENCY * (Uf @ P)
        self.Sy = -RHO_FUTURE_OUTPUT * (Yf @ P)

        Minv_UfT = la.cho_solve(M_factor, Uf.T, check_finite=False)
        Minv_YfT = la.cho_solve(M_factor, Yf.T, check_finite=False)
        Huu = np.diag(R_BAR) + LAMBDA_CONSISTENCY * np.eye(nu)
        Huu -= LAMBDA_CONSISTENCY**2 * (Uf @ Minv_UfT)
        Hyy = np.diag(Q_BAR) + RHO_FUTURE_OUTPUT * np.eye(ny)
        Hyy -= RHO_FUTURE_OUTPUT**2 * (Yf @ Minv_YfT)
        Huy = -LAMBDA_CONSISTENCY * RHO_FUTURE_OUTPUT * (Uf @ Minv_YfT)
        H = np.block([[Huu, Huy], [Huy.T, Hyy]])
        self.H = 0.5 * (H + H.T)

        y_low = np.r_[
            np.full(N_VEHICLES, VELOCITY_ERROR_MIN),
            np.full(N_CAV, SPACING_ERROR_MIN),
        ]
        y_high = np.r_[
            np.full(N_VEHICLES, VELOCITY_ERROR_MAX),
            np.full(N_CAV, SPACING_ERROR_MAX),
        ]
        lower = np.r_[np.full(nu, U_MIN), np.tile(y_low, HORIZON)]
        upper = np.r_[np.full(nu, U_MAX), np.tile(y_high, HORIZON)]
        self.center = 0.5 * (lower + upper)
        self.scale = 0.5 * (upper - lower)
        self.H_box = np.asfortranarray(
            self.scale[:, None] * self.H * self.scale[None, :]
        )
        self.h_center = self.H @ self.center
        self.dimension = len(self.center)
        self.x_warm = None

    def linear_term(self, uini, eini, yini, ef, reference):
        w = np.concatenate([
            np.asarray(uini).reshape(-1),
            np.asarray(eini).reshape(-1),
            np.asarray(yini).reshape(-1),
            np.asarray(ef).reshape(-1),
        ])
        hu = self.Su @ w
        hy = -Q_BAR * np.asarray(reference).reshape(-1) + self.Sy @ w
        return np.r_[hu, hy]

    def solve(self, uini, eini, yini, ef, reference):
        start = time.perf_counter()
        h = self.linear_term(uini, eini, yini, ef, reference)
        h_box = np.ascontiguousarray(
            self.scale * (self.h_center + h), dtype=np.float64
        )
        x = np.zeros(self.dimension, dtype=np.float64)
        info = BoxQPInfo()
        warm = None if self.x_warm is None else ptr(self.x_warm)
        status = lib.dense_boxqp_solve(
            ptr(self.H_box), ptr(h_box), BINT(self.dimension),
            ct.c_double(1e-7), BINT(60), warm, ptr(x), ct.byref(info),
        )
        elapsed = time.perf_counter() - start
        if status not in (0, 1):
            raise RuntimeError(f"C BoxQP status={status}, chol_info={info.chol_info}")

        self.x_warm = x.copy()
        z = self.center + self.scale * x
        nu = HORIZON * DIM_U
        return {
            "u": z[:nu].reshape(HORIZON, DIM_U),
            "y": z[nu:].reshape(HORIZON, DIM_Y),
            "time": elapsed,
            "iterations": int(info.iterations),
        }


## 6. Define one common closed-loop braking simulation


In [ ]:
# Closed-loop control comparison. The C controller and all-HDV baseline run first, so their results remain available even if the original hard-equality baseline is interrupted.

# Solve every 10 simulation steps.This value is shared by Original DeeP-LCC and C Fast-DeeP-LCC.
CONTROL_SHIFT = 10

# Repository-style automatic emergency braking (AEB). The optimization constrains predicted spacing; this independent safetylayer protects the nonlinear realized trajectory between replans.
AEB_SAFETY_MARGIN = 0.5
AEB_TTC_LIMIT = 1.0


def apply_aeb_safety_filter(plant, action, v_head):
    action = np.asarray(action, dtype=np.float64).reshape(-1).copy()
    activations = 0

    for control_index, vehicle_index in enumerate(plant.pos_cav):
        lead_velocity = float(
            v_head if vehicle_index == 0 else plant.v[vehicle_index - 1]
        )
        follower_velocity = float(plant.v[vehicle_index])
        spacing = float(plant.s[vehicle_index])
        closing_speed = max(0.0, follower_velocity - lead_velocity)

        # Available stopping distance before reaching the physical 5 m bound.
        available_distance = max(
            spacing - (S_ST + AEB_SAFETY_MARGIN), 1.0e-6
        )
        required_deceleration = max(
            0.0,
            (follower_velocity**2 - lead_velocity**2)
            / (2.0 * available_distance),
        )
        time_to_bound = (
            available_distance / closing_speed
            if closing_speed > 1.0e-9
            else np.inf
        )

        if (
            required_deceleration >= abs(U_MIN)
            or time_to_bound <= AEB_TTC_LIMIT
            or spacing <= S_ST + AEB_SAFETY_MARGIN
        ):
            action[control_index] = U_MIN
            activations += 1

    return action, activations


def run_closed_loop(
    mode,
    noise,
    head_velocity,
    show_progress=True,
):
    if mode not in {"all_hdv", "original", "soft", "fast_c"}:
        raise ValueError(
            "mode must be 'all_hdv', 'original', 'soft', or 'fast_c'"
        )

    vehicle_id = (
        np.zeros(N_VEHICLES, dtype=int)
        if mode == "all_hdv"
        else ID
    )

    plant = MixedTrafficPlatoon(vehicle_id)

    # Each controller is constructed only once, before the online loop.
    controller_class = {
        "original": OriginalDeepLCC,
        "soft": SoftDeepLCC,
        "fast_c": ReducedCDeepLCC,
    }.get(mode)
    controller = controller_class(BLOCKS) if controller_class else None

    u_window = np.zeros(
        (T_INI, DIM_U),
        dtype=np.float64,
    )
    e_window = np.zeros(
        (T_INI, DIM_E),
        dtype=np.float64,
    )
    y_window = np.zeros(
        (T_INI, DIM_Y),
        dtype=np.float64,
    )

    reference = np.zeros(
        HORIZON * DIM_Y,
        dtype=np.float64,
    )

    velocities = []
    spacings = []
    controls = []
    solve_times = []
    iterations = []
    aeb_activations = 0

    planned_actions = None
    number_of_solves = int(
        np.ceil(CONTROL_STEPS / CONTROL_SHIFT)
    )
    solve_number = 0

    for k in range(CONTROL_STEPS):
        if mode == "all_hdv":
            action = np.empty(0, dtype=np.float64)

        else:
            # Recompute the control plan only every CONTROL_SHIFT steps.
            if k % CONTROL_SHIFT == 0:
                solve_number += 1

                if show_progress:
                    print(
                        f"\r{mode}: solve "
                        f"{solve_number}/{number_of_solves}, "
                        f"simulation step {k}/{CONTROL_STEPS}",
                        end="",
                        flush=True,
                    )

                ef = (
                    head_velocity[k:k + HORIZON]
                    - V_STAR
                )

                if len(ef) < HORIZON:
                    ef = np.pad(
                        ef,
                        (0, HORIZON - len(ef)),
                        mode="edge",
                    )

                result = controller.solve(
                    u_window,
                    e_window,
                    y_window,
                    ef,
                    reference,
                )

                available_actions = min(
                    CONTROL_SHIFT,
                    len(result["u"]),
                )

                planned_actions = (
                    result["u"][:available_actions].copy()
                )

                # If CONTROL_SHIFT exceeds the prediction horizon,
                # repeat the last planned action.
                if available_actions < CONTROL_SHIFT:
                    planned_actions = np.vstack([
                        planned_actions,
                        np.repeat(
                            planned_actions[-1:, :],
                            CONTROL_SHIFT - available_actions,
                            axis=0,
                        ),
                    ])

                solve_times.append(result["time"])
                iterations.append(result["iterations"])

            action_index = k % CONTROL_SHIFT
            action = planned_actions[action_index].copy()

        if mode != "all_hdv":
            action, triggered = apply_aeb_safety_filter(
                plant, action, head_velocity[k]
            )
            aeb_activations += triggered

        measured = plant.step(
            action,
            head_velocity[k],
            noise[k],
        )

        velocities.append(plant.v.copy())
        spacings.append(plant.s.copy())

        if mode != "all_hdv":
            controls.append(action.copy())

            u_window = np.vstack([
                u_window[1:],
                action.reshape(1, -1),
            ])

            e_window = np.vstack([
                e_window[1:],
                [[head_velocity[k] - V_STAR]],
            ])

            y_window = np.vstack([
                y_window[1:],
                measured.reshape(1, -1),
            ])

    if show_progress and mode != "all_hdv":
        print()

    return {
        "velocity": np.asarray(
            velocities,
            dtype=np.float64,
        ),
        "spacing": np.asarray(
            spacings,
            dtype=np.float64,
        ),
        "control": np.asarray(
            controls,
            dtype=np.float64,
        ),
        "times": np.asarray(
            solve_times,
            dtype=np.float64,
        ),
        "iterations": np.asarray(
            iterations,
            dtype=np.float64,
        ),
        "aeb_activations": int(aeb_activations),
    }



## 7. Run all three controllers and report raw mean online solve times


In [ ]:
head_velocity, _ = brake_profile(CONTROL_STEPS)
rng = np.random.default_rng(SEED + 900)
shared_noise = rng.uniform(
    -ONLINE_NOISE, ONLINE_NOISE, size=(CONTROL_STEPS, N_VEHICLES)
)

print("Running original hard-equality DeeP-LCC...")
original_lcc = run_closed_loop("original", shared_noise, head_velocity)

print("Running full soft quadratic DeeP-LCC with OSQP...")
soft_lcc = run_closed_loop("soft", shared_noise, head_velocity)

print("Running T-independent C Fast-DeeP-LCC...")
fast_lcc = run_closed_loop("fast_c", shared_noise, head_velocity)

print("\nRAW MEAN ONLINE SOLVE TIMES (S)")
print(f"{'Controller':<34} {'Mean time (s)':>14}")
print("-" * 49)
for name, result in [
    ("Original hard-equality DeeP-LCC", original_lcc),
    ("Full soft quadratic OSQP", soft_lcc),
    ("T-independent C Fast-DeeP-LCC", fast_lcc),
]:
    print(f"{name:<34} {np.mean(result['times']):>14.6f}")

## 8. Plot CAV spacing 


In [ ]:
# Compare the realized spacing of CAVs 3 and 6.
number_of_steps = min(
    len(original_lcc["spacing"]),
    len(fast_lcc["spacing"]),
)
time_s = np.arange(number_of_steps) * DT

fig, axes = plt.subplots(
    1, 2, figsize=(11, 3.8), sharex=True, sharey=True
)

for axis, vehicle in zip(axes, POS_CAV):
    axis.plot(
        time_s,
        original_lcc["spacing"][:number_of_steps, vehicle],
        color="tab:blue",
        linewidth=2.4,
        label="Original DeeP-LCC",
    )
    axis.plot(
        time_s,
        fast_lcc["spacing"][:number_of_steps, vehicle],
        color="tab:orange",
        linestyle="--",
        linewidth=2.4,
        label="C Fast-DeeP-LCC",
    )
    axis.axhline(5.0, color="red", linestyle=":", linewidth=1.6,
                 label="Spacing bounds")
    axis.axhline(40.0, color="red", linestyle=":", linewidth=1.6)
    axis.set_title(f"CAV {vehicle + 1} spacing")
    axis.set_xlabel("Time (s)")
    axis.grid(True, alpha=0.3)

axes[0].set_ylabel("Spacing (m)")
axes[0].legend(fontsize=10, loc="center right")

fig.tight_layout()
spacing_plot_path = ROOT / "media" / "mixed_traffic_cav_spacing.png"
fig.savefig(spacing_plot_path, dpi=250, bbox_inches="tight")
plt.show()
print("Saved spacing plot:", spacing_plot_path)


## 8. Write the mixed-traffic controller-comparison GIF


In [ ]:
# Animated mixed-traffic platoon comparison.
# Uses all_hdv, fast_lcc, and optionally original_lcc results.

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from IPython.display import Image, display


FRAME_STRIDE = 2
GIF_FPS = 20
CAR_LENGTH = 4.5
CAR_HEIGHT = 1.15
VIEW_BEHIND_HEAD = 185.0
VIEW_AHEAD_HEAD = 18.0

gif_path = ROOT / "media" / "mixed_traffic.gif"


# Include every successfully completed simulation.
scenarios = [
    ("Original hard-equality DeeP-LCC", original_lcc),
    ("C Fast-DeeP-LCC BoxQP", fast_lcc),
]


def reconstruct_vehicle_positions(result):
    """
    Reconstruct longitudinal positions from vehicle spacing.

    Vehicle 0 is the external head vehicle. Vehicles 1-8 are the
    following mixed-traffic vehicles.
    """
    spacing = np.asarray(
        result["spacing"],
        dtype=np.float64,
    )

    number_of_steps = len(spacing)

    head_speed = np.asarray(
        head_velocity[:number_of_steps],
        dtype=np.float64,
    )

    # Integrate the head-vehicle velocity.
    head_position = np.cumsum(
        head_speed * DT
    )

    # Each spacing is the distance from one vehicle to its leader.
    following_positions = (
        head_position[:, None]
        - np.cumsum(spacing, axis=1)
    )

    return np.column_stack([
        head_position,
        following_positions,
    ])


# Precompute all positions.
for index, (name, result) in enumerate(scenarios):
    scenarios[index] = (
        name,
        result,
        reconstruct_vehicle_positions(result),
    )


maximum_steps = max(
    len(positions)
    for _, _, positions in scenarios
)

frame_steps = list(
    range(0, maximum_steps, FRAME_STRIDE)
)

if frame_steps[-1] != maximum_steps - 1:
    frame_steps.append(maximum_steps - 1)


# Vehicle colors.
HEAD_COLOR = "#202020"
HDV_COLOR = "#8c8c8c"
CAV_COLOR = "#f28e2b"
CAV_EDGE = "#9a4b00"
HDV_EDGE = "#404040"

number_of_panels = len(scenarios)

with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
}):
    fig, axes = plt.subplots(
        number_of_panels,
        1,
        figsize=(12, 2.65 * number_of_panels),
        squeeze=False,
    )

    axes = axes[:, 0]
    animated_panels = []

    # Determine the complete world range for stationary road markings.
    largest_head_distance = max(
        positions[-1, 0]
        for _, _, positions in scenarios
    )

    world_minimum = -220.0
    world_maximum = largest_head_distance + 40.0

    for axis, (name, result, positions) in zip(
        axes,
        scenarios,
    ):
        axis.set_facecolor("#dceeff")

        # Road.
        axis.add_patch(
            Rectangle(
                (world_minimum, -2.35),
                world_maximum - world_minimum,
                4.70,
                facecolor="#4c4c4c",
                edgecolor="#202020",
                linewidth=1.2,
                zorder=0,
            )
        )

        # Road boundaries.
        axis.plot(
            [world_minimum, world_maximum],
            [-2.12, -2.12],
            color="white",
            linewidth=1.5,
            zorder=1,
        )
        axis.plot(
            [world_minimum, world_maximum],
            [2.12, 2.12],
            color="white",
            linewidth=1.5,
            zorder=1,
        )

        # Moving-world lane markings.
        for marker_position in np.arange(
            world_minimum,
            world_maximum,
            12.0,
        ):
            axis.plot(
                [
                    marker_position,
                    marker_position + 6.0,
                ],
                [0.0, 0.0],
                color="white",
                linewidth=1.3,
                alpha=0.75,
                zorder=1,
            )

        vehicle_patches = []
        vehicle_labels = []

        # Head vehicle plus eight following vehicles.
        for vehicle_index in range(N_VEHICLES + 1):
            if vehicle_index == 0:
                face_color = HEAD_COLOR
                edge_color = "black"
                label = "Head"
            else:
                platoon_index = vehicle_index - 1

                if ID[platoon_index] == 1:
                    face_color = CAV_COLOR
                    edge_color = CAV_EDGE
                    label = f"CAV {vehicle_index}"
                else:
                    face_color = HDV_COLOR
                    edge_color = HDV_EDGE
                    label = f"HDV {vehicle_index}"

            initial_x = positions[0, vehicle_index]

            vehicle_patch = Rectangle(
                (
                    initial_x - 0.5 * CAR_LENGTH,
                    -0.5 * CAR_HEIGHT,
                ),
                CAR_LENGTH,
                CAR_HEIGHT,
                facecolor=face_color,
                edgecolor=edge_color,
                linewidth=1.4,
                zorder=5,
            )

            axis.add_patch(vehicle_patch)
            vehicle_patches.append(vehicle_patch)

            vehicle_text = axis.text(
                initial_x,
                0.92,
                label,
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="black",
                zorder=7,
            )

            vehicle_labels.append(vehicle_text)

        # Legend entries.
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=HEAD_COLOR,
            edgecolor="black",
            label="Head vehicle",
        )
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=HDV_COLOR,
            edgecolor=HDV_EDGE,
            label="HDV",
        )
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=CAV_COLOR,
            edgecolor=CAV_EDGE,
            label="CAV",
        )

        information_text = axis.text(
            0.985,
            0.92,
            "",
            transform=axis.transAxes,
            ha="right",
            va="top",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.28",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92,
            ),
            zorder=10,
        )

        axis.set_title(name)
        axis.set_ylim(-2.8, 2.8)
        axis.set_yticks([])
        axis.set_ylabel("Road")
        axis.grid(False)

        initial_head = positions[0, 0]
        axis.set_xlim(
            initial_head - VIEW_BEHIND_HEAD,
            initial_head + VIEW_AHEAD_HEAD,
        )

        animated_panels.append({
            "result": result,
            "positions": positions,
            "patches": vehicle_patches,
            "labels": vehicle_labels,
            "text": information_text,
        })

    axes[-1].set_xlabel(
        "Longitudinal road position (m)"
    )

    axes[0].legend(
        loc="lower left",
        ncol=3,
        fontsize=9,
        framealpha=0.95,
    )

    fig.suptitle(
        "Mixed-traffic platoon during the head-vehicle braking disturbance",
        fontsize=15,
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.985,
        bottom=0.09,
        top=0.91,
        hspace=0.33,
    )


    def update(frame_number):
        simulation_step = frame_steps[frame_number]
        artists = []

        for axis, panel in zip(
            axes,
            animated_panels,
        ):
            positions = panel["positions"]
            result = panel["result"]

            index = min(
                simulation_step,
                len(positions) - 1,
            )

            current_positions = positions[index]
            current_head_position = current_positions[0]

            # Scroll the viewing window with the head vehicle.
            axis.set_xlim(
                current_head_position - VIEW_BEHIND_HEAD,
                current_head_position + VIEW_AHEAD_HEAD,
            )

            for vehicle_index, (
                vehicle_patch,
                vehicle_label,
            ) in enumerate(zip(
                panel["patches"],
                panel["labels"],
            )):
                x_position = current_positions[vehicle_index]

                vehicle_patch.set_xy((
                    x_position - 0.5 * CAR_LENGTH,
                    -0.5 * CAR_HEIGHT,
                ))

                vehicle_label.set_position((
                    x_position,
                    0.92,
                ))

                artists.extend([
                    vehicle_patch,
                    vehicle_label,
                ])

            time_seconds = index * DT

            # Current minimum CAV spacing.
            current_spacing = np.asarray(
                result["spacing"][index],
                dtype=np.float64,
            )

            minimum_cav_spacing = np.min(
                current_spacing[POS_CAV]
            )

            panel["text"].set_text(
                f"time = {time_seconds:.2f} s\n"
                f"step = {index}\n"
                f"minimum CAV spacing = "
                f"{minimum_cav_spacing:.2f} m"
            )

            artists.append(panel["text"])

        return artists


    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_steps),
        interval=1000 / GIF_FPS,
        blit=False,
        repeat=True,
    )

    animation.save(
        gif_path,
        writer=PillowWriter(fps=GIF_FPS),
        dpi=120,
    )

    plt.close(fig)


print("Saved GIF:", gif_path)
